# Spaceship Titanic - Model Tuning

This notebook improves on the baseline by:
- Reusing the cleaning/parsing pipeline
- Benchmarking two stronger model families
- Running compact hyperparameter searches
- Selecting the best CV model
- Exporting a tuned `submission_tuned.csv`


In [7]:
# Standard library path helper.
from pathlib import Path

# Core numerical/data libraries.
import numpy as np
import pandas as pd

# Modeling and tuning utilities.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


In [8]:
# Resolve project root robustly for notebook vs repo-root execution.
base_dir = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

# Load training and inference datasets.
train = pd.read_csv(base_dir / 'train.csv')
test = pd.read_csv(base_dir / 'test.csv')

# Quick shape sanity check.
print('Train shape:', train.shape)
print('Test shape:', test.shape)


Train shape: (8693, 14)
Test shape: (4277, 13)


## Cleaning and Parsing

In [9]:
def clean_and_parse(df: pd.DataFrame) -> pd.DataFrame:
    # Copy first so source DataFrame is preserved.
    df = df.copy()

    # Parse PassengerId into group and member components.
    pid = df['PassengerId'].astype(str).str.split('_', n=1, expand=True)
    df['GroupId'] = pid[0]
    df['GroupMemberId'] = pd.to_numeric(pid[1], errors='coerce')

    # Parse Cabin into deck / number / side.
    cabin = df['Cabin'].astype(str).str.split('/', n=2, expand=True)
    df['CabinDeck'] = cabin[0].replace('nan', np.nan)
    df['CabinNum'] = pd.to_numeric(cabin[1], errors='coerce')
    df['CabinSide'] = cabin[2].replace('nan', np.nan)

    # Parse Name into first and last name tokens.
    name = df['Name'].astype(str).str.split(' ', n=1, expand=True)
    df['FirstName'] = name[0].replace('nan', np.nan)
    df['LastName'] = name[1].replace('nan', np.nan)

    # Prepare spending columns and derive spend-based features.
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    for c in spend_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    df['TotalSpend'] = df[spend_cols].fillna(0).sum(axis=1)
    df['NoSpend'] = (df['TotalSpend'] == 0).astype(int)
    df['LogTotalSpend'] = np.log1p(df['TotalSpend'])

    # Create an age bucket categorical feature.
    df['AgeBucket'] = pd.cut(
        pd.to_numeric(df['Age'], errors='coerce'),
        bins=[-1, 12, 18, 25, 40, 60, 200],
        labels=['child', 'teen', 'young_adult', 'adult', 'mid_age', 'senior']
    )

    # Interaction feature between origin and destination planets.
    df['HomePlanet_Destination'] = (
        df['HomePlanet'].astype(str).replace('nan', 'Unknown') + '_' +
        df['Destination'].astype(str).replace('nan', 'Unknown')
    )

    # Group-level context features.
    df['FamilyNameSize'] = df.groupby('LastName')['PassengerId'].transform('count')
    df['GroupSize'] = df.groupby('GroupId')['PassengerId'].transform('count')

    # Normalize booleans as strings for sklearn compatibility across versions.
    for c in ['CryoSleep', 'VIP']:
        df[c] = df[c].map({True: 'True', False: 'False'})

    # Ensure target dtype consistency for train data.
    if 'Transported' in df.columns:
        df['Transported'] = df['Transported'].astype(bool)

    return df


In [10]:
# Clean and parse raw train/test data.
train_clean = clean_and_parse(train)
test_clean = clean_and_parse(test)

# Define target and drop non-modeling ID/raw text fields.
target_col = 'Transported'
drop_cols = ['PassengerId', 'Name', 'Cabin']

# Build feature matrices and labels.
X = train_clean.drop(columns=[target_col]).drop(columns=drop_cols, errors='ignore')
y = train_clean[target_col].astype(int)
X_test = test_clean.drop(columns=drop_cols, errors='ignore')

# Ensure bool-like columns are plain object categories before preprocessing.
for c in ['CryoSleep', 'VIP']:
    if c in X.columns:
        X[c] = X[c].astype(object)
    if c in X_test.columns:
        X_test[c] = X_test[c].astype(object)

# Infer numeric vs categorical/boolean columns for preprocessing.
cat_cols = X.select_dtypes(include=['object', 'category', 'boolean']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

# Print feature-space summary for sanity.
print('Feature matrix shape:', X.shape)
print('Numeric cols:', len(num_cols), '| Categorical/Boolean cols:', len(cat_cols))


Feature matrix shape: (8693, 24)
Numeric cols: 13 | Categorical/Boolean cols: 11


In [11]:
# Shared preprocessing used across all model comparisons.
preprocessor = ColumnTransformer(
    transformers=[
        # Numeric: median imputation for missing values.
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols),
        # Categorical: most-frequent imputation then one-hot encode.
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            # HGB requires dense numeric arrays, so force dense one-hot output.
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols),
    ]
)

# Stratified CV keeps class ratio stable in each fold.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## Quick Benchmark Before Search

In [12]:
# Baseline RF pipeline for quick benchmark.
rf_base = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1))
])

# Baseline HGB pipeline for quick benchmark.
hgb_base = Pipeline([
    ('preprocess', preprocessor),
    ('model', HistGradientBoostingClassifier(random_state=42))
])

# Compare both models with identical cross-validation protocol.
rf_scores = cross_val_score(rf_base, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
hgb_scores = cross_val_score(hgb_base, X, y, cv=cv, scoring='accuracy', n_jobs=-1)

print('RF CV mean:', round(rf_scores.mean(), 5), '+/-', round(rf_scores.std(), 5))
print('HGB CV mean:', round(hgb_scores.mean(), 5), '+/-', round(hgb_scores.std(), 5))


RF CV mean: 0.78431 +/- 0.00742
HGB CV mean: 0.80628 +/- 0.00727


## Random Forest Search

In [20]:
# RF pipeline to be tuned via randomized search.
rf_pipe = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(random_state=42, n_jobs=4))
])

# Compact but meaningful RF hyperparameter space.
rf_param_dist = {
    'model__n_estimators': [300, 500, 700, 900],
    'model__max_depth': [None, 10, 15, 20, 30],
    'model__min_samples_split': [2, 5, 10, 20],
    'model__min_samples_leaf': [1, 2, 4, 8],
    'model__max_features': ['sqrt', 'log2', None],
}

# Randomized search balances exploration quality and runtime.
rf_search = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=rf_param_dist,
    n_iter=18,
    scoring='accuracy',
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=1,
    refit=True
)

# Run search and report best RF configuration.
rf_search.fit(X, y)
print('RF best score:', round(rf_search.best_score_, 5))
print('RF best params:', rf_search.best_params_)


Fitting 5 folds for each of 18 candidates, totalling 90 fits
RF best score: 0.80306
RF best params: {'model__n_estimators': 900, 'model__min_samples_split': 10, 'model__min_samples_leaf': 8, 'model__max_features': None, 'model__max_depth': 30}


/opt/homebrew/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


## HistGradientBoosting Search

In [21]:
# HGB pipeline to be tuned via randomized search.
hgb_pipe = Pipeline([
    ('preprocess', preprocessor),
    ('model', HistGradientBoostingClassifier(random_state=42))
])

# Hyperparameter space for boosting learning dynamics and regularization.
hgb_param_dist = {
    'model__learning_rate': [0.02, 0.03, 0.05, 0.08, 0.1],
    'model__max_iter': [150, 250, 350, 500],
    'model__max_leaf_nodes': [15, 31, 63, 127],
    'model__min_samples_leaf': [10, 20, 30, 40],
    'model__l2_regularization': [0.0, 0.01, 0.1, 1.0],
}

# Randomized search for HGB model family.
hgb_search = RandomizedSearchCV(
    estimator=hgb_pipe,
    param_distributions=hgb_param_dist,
    n_iter=18,
    scoring='accuracy',
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=4,
    refit=True
)

# Run search and report best HGB configuration.
hgb_search.fit(X, y)
print('HGB best score:', round(hgb_search.best_score_, 5))
print('HGB best params:', hgb_search.best_params_)


Fitting 5 folds for each of 18 candidates, totalling 90 fits
HGB best score: 0.81341
HGB best params: {'model__min_samples_leaf': 30, 'model__max_leaf_nodes': 15, 'model__max_iter': 150, 'model__learning_rate': 0.05, 'model__l2_regularization': 0.1}


## Select Winner and Export Submission

In [ ]:
# Choose the model with the best cross-validated accuracy.
best_name = 'RandomForest' if rf_search.best_score_ >= hgb_search.best_score_ else 'HistGradientBoosting'
best_model = rf_search.best_estimator_ if best_name == 'RandomForest' else hgb_search.best_estimator_
best_score = max(rf_search.best_score_, hgb_search.best_score_)

print('Selected model:', best_name)
print('Selected CV score:', round(best_score, 5))

# Refit winning model on full training set and predict test set.
best_model.fit(X, y)
test_preds = best_model.predict(X_test).astype(bool)

# Create output in required competition submission format.
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': test_preds
})

# Save tuned submission and preview first rows.
out_path = base_dir / 'submission_tuned.csv'
submission.to_csv(out_path, index=False)
print(f'Saved tuned submission to: {out_path}')
submission.head()
